# Build `maths_data` from raw AM, Adaptiv World, and Adaptiv College attempts

This notebook reads the original AM attempt file from `data_am_mia/` and the Adaptiv World and Adaptiv College attempt files from their source folders. It removes the same excluded work modes as the original build, keeps the selected maths modules, derives retry and session fields, converts absolute timestamps to session-relative timestamps, applies the existing adaptive-transition filters, and writes `data_miaam/maths_data.parquet`.

The source labels in the merged dataset are `am`, `adaptiv_world`, and `adaptiv_college`. The notebook intentionally preserves the original pipeline behavior and does not add a raw-row deduplication step.

## Step 1: Imports and constants

Load the libraries used in the notebook and define the key input paths, selected module scopes, shared column lists, and timestamp parsing settings.


In [1]:
import json
from pathlib import Path

import polars as pl


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repo root from the current working directory.")


def find_single_csv(directory: Path) -> Path:
    csv_paths = sorted(directory.glob("*.csv"))
    if len(csv_paths) != 1:
        raise RuntimeError(f"Expected exactly one CSV in {directory}, found {len(csv_paths)}.")
    return csv_paths[0]


ROOT = find_repo_root()
AM_SOURCE_DIR = ROOT / "data_am_mia"
ADAPTIV_WORLD_DIR = ROOT / "data_adaptiv_world"
ADAPTIV_COLLEGE_DIR = ROOT / "data_adaptiv_college"
OUTPUT_DIR = ROOT / "data_miaam"

AM_RAW_PATH = AM_SOURCE_DIR / "am.csv"
AM_LEARNING_CATALOG_PATH = AM_SOURCE_DIR / "learning_catalog.json"

ADAPTIV_WORLD_RAW_PATH = find_single_csv(ADAPTIV_WORLD_DIR)
ADAPTIV_WORLD_CONFIG_PATH = ADAPTIV_WORLD_DIR / "data_AdaptivWorld.json"
ADAPTIV_WORLD_GRAPH_PATH = ADAPTIV_WORLD_DIR / "graph_AdaptivWorld.json"

ADAPTIV_COLLEGE_RAW_PATH = find_single_csv(ADAPTIV_COLLEGE_DIR)
ADAPTIV_COLLEGE_CONFIG_PATH = ADAPTIV_COLLEGE_DIR / "data_AdaptivCollege.json"
ADAPTIV_COLLEGE_GRAPH_PATH = ADAPTIV_COLLEGE_DIR / "graph_AdaptivCollege.json"

MATHS_DATA_PATH = OUTPUT_DIR / "maths_data.parquet"

PREP_COLUMNS = [
    "user_id",
    "classroom_id",
    "playlist_or_module_id",
    "exercise_id",
    "created_at",
    "login_time",
    "data_correct",
    "work_mode",
    "data_answer",
    "data_duration",
    "source",
]

INTERMEDIATE_COLUMNS = [
    "user_id",
    "classroom_id",
    "playlist_or_module_id",
    "exercise_id",
    "created_at",
    "login_time",
    "data_correct",
    "work_mode",
    "data_answer",
    "data_duration",
    "source",
    "attempt_index",
    "session_id",
]

FINAL_COLUMNS = [
    "user_id",
    "classroom_id",
    "playlist_or_module_id",
    "exercise_id",
    "created_at",
    "data_correct",
    "work_mode",
    "data_answer",
    "data_duration",
    "source",
    "attempt_index",
    "session_id",
]

SOURCE_LABELS = ("am", "adaptiv_world", "adaptiv_college")
EXCLUDED_WORK_MODES = ["initial-test", "duo", "revision"]
AM_SCOPED_MODULE_CODES = ("M1", "M31", "M32", "M33")
ADAPTIV_WORLD_SCOPED_MODULE_CODES = ("M101", "M103")
ADAPTIV_COLLEGE_SCOPED_MODULE_CODES = ("M101", "M102", "M103")
TIMESTAMP_FORMAT = "%Y-%m-%d %H:%M:%S%.f%:z"


def sink_parquet_replace(lf: pl.LazyFrame, path: Path, *, compression: str = "zstd") -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_name(f"{path.name}.tmp")
    tmp_path.unlink(missing_ok=True)
    lf.sink_parquet(tmp_path, compression=compression)
    tmp_path.replace(path)

## Step 2: Load the raw attempt files

Open the AM, Adaptiv World, and Adaptiv College raw CSV files lazily and inspect their available columns before applying any filtering.

In [2]:
am_raw = pl.scan_csv(AM_RAW_PATH, infer_schema_length=2000)
adaptiv_world_raw = pl.scan_csv(ADAPTIV_WORLD_RAW_PATH, infer_schema_length=2000)
adaptiv_college_raw = pl.scan_csv(ADAPTIV_COLLEGE_RAW_PATH, infer_schema_length=2000)

print(f"AM raw path: {AM_RAW_PATH}")
print(f"Adaptiv World raw path: {ADAPTIV_WORLD_RAW_PATH}")
print(f"Adaptiv College raw path: {ADAPTIV_COLLEGE_RAW_PATH}")
print("AM columns:", am_raw.collect_schema().names())
print("Adaptiv World columns:", adaptiv_world_raw.collect_schema().names())
print("Adaptiv College columns:", adaptiv_college_raw.collect_schema().names())

AM raw path: C:\Users\ocler\Documents\Académique\Inria\GAIMHE\Code\visu2\data_am_mia\am.csv
Adaptiv World raw path: C:\Users\ocler\Documents\Académique\Inria\GAIMHE\Code\visu2\data_adaptiv_world\1319-neurips-adaptiv-world_20260727_142625.csv
Adaptiv College raw path: C:\Users\ocler\Documents\Académique\Inria\GAIMHE\Code\visu2\data_adaptiv_college\1321-neurips-adaptiv-college_20260727_141220.csv
AM columns: ['UAI', 'classroom_id', 'teacher_id', 'user_id', 'playlist_or_module_id', 'objective_id', 'activity_id', 'exercise_id', 'module_short_title', 'module_long_title', 'created_at', 'login_time', 'is_initial_test', 'data_score', 'data_correct', 'data_nb_tries', 'work_mode', 'data_answer', 'data_duration', 'session_duration']
Adaptiv World columns: ['UAI', 'classroom_id', 'teacher_id', 'user_id', 'playlist_or_module_id', 'objective_id', 'activity_id', 'exercise_id', 'module_short_title', 'module_long_title', 'created_at', 'login_time', 'is_initial_test', 'data_score', 'data_correct', 'data

## Step 3: Remove excluded work modes

Drop work modes that should not be kept in the merged dataset and print the remaining work modes for each source.


In [3]:
am_filtered = am_raw.filter(
    ~pl.col("work_mode").cast(pl.Utf8, strict=False).is_in(EXCLUDED_WORK_MODES)
)
adaptiv_world_filtered = adaptiv_world_raw.filter(
    ~pl.col("work_mode").cast(pl.Utf8, strict=False).is_in(EXCLUDED_WORK_MODES)
)
adaptiv_college_filtered = adaptiv_college_raw.filter(
    ~pl.col("work_mode").cast(pl.Utf8, strict=False).is_in(EXCLUDED_WORK_MODES)
)


def retained_work_modes(lf: pl.LazyFrame) -> list[str]:
    return (
        lf.select(pl.col("work_mode").drop_nulls().unique().sort()).collect().to_series().to_list()
    )


print("Excluded work modes:", EXCLUDED_WORK_MODES)
print("AM retained work modes:", retained_work_modes(am_filtered))
print("Adaptiv World retained work modes:", retained_work_modes(adaptiv_world_filtered))
print("Adaptiv College retained work modes:", retained_work_modes(adaptiv_college_filtered))

Excluded work modes: ['initial-test', 'duo', 'revision']
AM retained work modes: ['adaptive-test', 'playlist', 'zpdes']
Adaptiv World retained work modes: ['adaptive-test', 'playlist', 'zpdes']
Adaptiv College retained work modes: ['adaptive-test', 'zpdes']


## Step 4: Keep only the selected module subsets

Resolve the module ids for the chosen AM, Adaptiv World, and Adaptiv College maths scopes. For AM, keep rows whose `playlist_or_module_id` matches the selected module ids. For both Adaptiv sources, keep rows whose `playlist_or_module_id` matches the selected module ids and retain playlist-like rows whose `exercise_id` belongs to a selected module.

In [4]:
am_learning_catalog = json.loads(AM_LEARNING_CATALOG_PATH.read_text(encoding="utf-8"))

adaptiv_world_config = json.loads(ADAPTIV_WORLD_CONFIG_PATH.read_text(encoding="utf-8"))["config"]
adaptiv_world_graph = json.loads(ADAPTIV_WORLD_GRAPH_PATH.read_text(encoding="utf-8"))

adaptiv_college_config = json.loads(ADAPTIV_COLLEGE_CONFIG_PATH.read_text(encoding="utf-8"))[
    "config"
]
adaptiv_college_graph = json.loads(ADAPTIV_COLLEGE_GRAPH_PATH.read_text(encoding="utf-8"))


def metadata_id_code_map(values) -> dict[str, str]:
    return {str(value["id"]): str(value.get("code") or "") for value in values if value.get("id")}


def validate_config_graph_consistency(
    config: dict,
    graph: dict,
    *,
    source_label: str,
) -> None:
    config_modules = metadata_id_code_map(config["module"].values())
    config_objectives = metadata_id_code_map(config["objective"].values())
    config_activities = metadata_id_code_map(config["activity"].values())

    graph_modules = metadata_id_code_map(graph["modules"])
    graph_objectives = metadata_id_code_map(graph["objectives"])
    graph_activities = metadata_id_code_map(graph["activities"])

    assert config_modules == graph_modules, f"{source_label}: module metadata mismatch"
    assert config_objectives == graph_objectives, f"{source_label}: objective metadata mismatch"
    assert config_activities == graph_activities, f"{source_label}: activity metadata mismatch"

    config_exercises_by_activity = {
        str(activity["id"]): {
            str(exercise_id)
            for exercise_id in (activity.get("learning_items") or [])
            if exercise_id
        }
        for activity in config["activity"].values()
    }
    graph_exercises_by_activity = {
        str(activity["id"]): {
            str(exercise_id) for exercise_id in (activity.get("exerciseIds") or []) if exercise_id
        }
        for activity in graph["activities"]
    }
    assert config_exercises_by_activity == graph_exercises_by_activity, (
        f"{source_label}: activity-to-exercise metadata mismatch"
    )


def collect_graph_scope(
    graph: dict,
    module_codes: tuple[str, ...],
) -> tuple[list[str], list[str]]:
    selected_modules = [
        module for module in graph["modules"] if str(module.get("code")) in module_codes
    ]
    selected_module_ids = {str(module["id"]) for module in selected_modules}
    selected_objective_ids = {
        str(objective_id)
        for module in selected_modules
        for objective_id in (module.get("objectiveIds") or [])
        if objective_id
    }
    selected_activity_ids = {
        str(activity_id)
        for objective in graph["objectives"]
        if str(objective.get("id")) in selected_objective_ids
        for activity_id in (objective.get("activityIds") or [])
        if activity_id
    }
    selected_exercise_ids = {
        str(exercise_id)
        for activity in graph["activities"]
        if str(activity.get("id")) in selected_activity_ids
        for exercise_id in (activity.get("exerciseIds") or [])
        if exercise_id
    }
    return sorted(selected_module_ids), sorted(selected_exercise_ids)


validate_config_graph_consistency(
    adaptiv_world_config,
    adaptiv_world_graph,
    source_label="adaptiv_world",
)
validate_config_graph_consistency(
    adaptiv_college_config,
    adaptiv_college_graph,
    source_label="adaptiv_college",
)

am_allowed_module_ids = sorted(
    str(module["id"])
    for module in am_learning_catalog["modules"]
    if str(module.get("code")) in AM_SCOPED_MODULE_CODES
)
adaptiv_world_allowed_module_ids, adaptiv_world_scoped_exercise_ids = collect_graph_scope(
    adaptiv_world_graph,
    ADAPTIV_WORLD_SCOPED_MODULE_CODES,
)
adaptiv_college_allowed_module_ids, adaptiv_college_scoped_exercise_ids = collect_graph_scope(
    adaptiv_college_graph,
    ADAPTIV_COLLEGE_SCOPED_MODULE_CODES,
)

am_scoped = am_filtered.filter(
    pl.col("playlist_or_module_id").cast(pl.Utf8, strict=False).is_in(am_allowed_module_ids)
)
adaptiv_world_scoped = adaptiv_world_filtered.filter(
    pl.col("playlist_or_module_id")
    .cast(pl.Utf8, strict=False)
    .is_in(adaptiv_world_allowed_module_ids)
    | pl.col("exercise_id").cast(pl.Utf8, strict=False).is_in(adaptiv_world_scoped_exercise_ids)
)
adaptiv_college_scoped = adaptiv_college_filtered.filter(
    pl.col("playlist_or_module_id")
    .cast(pl.Utf8, strict=False)
    .is_in(adaptiv_college_allowed_module_ids)
    | pl.col("exercise_id").cast(pl.Utf8, strict=False).is_in(adaptiv_college_scoped_exercise_ids)
)


def lazy_row_count(lf: pl.LazyFrame) -> int:
    return lf.select(pl.len().alias("rows")).collect().item(0, "rows")


print("AM scoped module codes:", list(AM_SCOPED_MODULE_CODES))
print("AM scoped module ids:", am_allowed_module_ids)
print("Adaptiv World scoped module codes:", list(ADAPTIV_WORLD_SCOPED_MODULE_CODES))
print("Adaptiv World scoped module ids:", adaptiv_world_allowed_module_ids)
print("Adaptiv World scoped exercise ids:", len(adaptiv_world_scoped_exercise_ids))
print("Adaptiv College scoped module codes:", list(ADAPTIV_COLLEGE_SCOPED_MODULE_CODES))
print("Adaptiv College scoped module ids:", adaptiv_college_allowed_module_ids)
print("Adaptiv College scoped exercise ids:", len(adaptiv_college_scoped_exercise_ids))
print("AM rows after module filter:", lazy_row_count(am_scoped))
print("Adaptiv World rows after module-aware filter:", lazy_row_count(adaptiv_world_scoped))
print("Adaptiv College rows after module-aware filter:", lazy_row_count(adaptiv_college_scoped))

AM scoped module codes: ['M1', 'M31', 'M32', 'M33']
AM scoped module ids: ['14fe4ca0-8fff-4c4a-bad2-6ef051eee349', '27709aa2-b055-4ed3-ac73-8dca783b4afe', '63e98e5f-94e3-4630-9704-076882d6de38', '8ff53d40-9b1f-44c8-8646-f699fed002e9']
Adaptiv World scoped module codes: ['M101', 'M103']
Adaptiv World scoped module ids: ['053df3ec-5501-4ad8-9917-a935bcf76740', '14321a7e-4ef7-4b6a-9ff8-99329e08d7a2']
Adaptiv World scoped exercise ids: 1527
Adaptiv College scoped module codes: ['M101', 'M102', 'M103']
Adaptiv College scoped module ids: ['1977213e-f43b-407c-a455-488c15445417', '6075b1c1-8edb-4d6a-9524-76d91f86de10', '9c85b221-0536-4863-a69a-d8c42f9323c2']
Adaptiv College scoped exercise ids: 2037
AM rows after module filter: 5900089
Adaptiv World rows after module-aware filter: 251694
Adaptiv College rows after module-aware filter: 62650


## Step 5: Remove students with excessive retries, except on single-exercise activities

Identify all single-exercise activities in the selected AM, Adaptiv World, and Adaptiv College modules, exempt those exercises from the retry cap, then remove any remaining source-specific student history where one exercise goes above `20` retries. Print the number and percentage of removed students and attempts.

In [5]:
MAX_RETRIES_PER_EXERCISE = 20
MAX_TOTAL_ATTEMPTS_PER_EXERCISE = MAX_RETRIES_PER_EXERCISE + 1


def collect_am_single_exercise_metadata(
    catalog: dict,
    module_codes: tuple[str, ...],
) -> tuple[set[str], list[str]]:
    exempt_exercise_ids: set[str] = set()
    exempt_activity_codes: list[str] = []
    for module in catalog["modules"]:
        if str(module.get("code")) not in module_codes:
            continue
        for objective in module.get("objectives", []):
            for activity in objective.get("activities", []):
                exercise_ids = [
                    str(exercise_id)
                    for exercise_id in activity.get("exercise_ids", [])
                    if exercise_id
                ]
                if len(exercise_ids) == 1:
                    exempt_exercise_ids.update(exercise_ids)
                    exempt_activity_codes.append(str(activity.get("code")))
    return exempt_exercise_ids, sorted(exempt_activity_codes)


def collect_graph_single_exercise_metadata(
    graph: dict,
    module_codes: tuple[str, ...],
) -> tuple[set[str], list[str]]:
    selected_module_ids = {
        str(module["id"]) for module in graph["modules"] if str(module.get("code")) in module_codes
    }
    selected_objective_ids = {
        str(objective_id)
        for module in graph["modules"]
        if str(module.get("id")) in selected_module_ids
        for objective_id in (module.get("objectiveIds") or [])
        if objective_id
    }
    selected_activity_ids = {
        str(activity_id)
        for objective in graph["objectives"]
        if str(objective.get("id")) in selected_objective_ids
        for activity_id in (objective.get("activityIds") or [])
        if activity_id
    }

    exempt_exercise_ids: set[str] = set()
    exempt_activity_codes: list[str] = []
    for activity in graph["activities"]:
        if str(activity.get("id")) not in selected_activity_ids:
            continue
        exercise_ids = [
            str(exercise_id) for exercise_id in (activity.get("exerciseIds") or []) if exercise_id
        ]
        if len(exercise_ids) == 1:
            exempt_exercise_ids.update(exercise_ids)
            exempt_activity_codes.append(str(activity.get("code")))
    return exempt_exercise_ids, sorted(exempt_activity_codes)


def apply_retry_cap(
    lf: pl.LazyFrame,
    *,
    source_label: str,
    exempt_exercise_ids: set[str],
) -> tuple[pl.LazyFrame, dict[str, int | float | str]]:
    normalized = lf.with_columns(
        pl.col("user_id").cast(pl.Utf8, strict=False).alias("__user_id"),
        pl.col("exercise_id").cast(pl.Utf8, strict=False).alias("__exercise_id"),
    )

    offending_users = (
        normalized.select("__user_id", "__exercise_id")
        .filter(pl.col("__user_id").is_not_null() & pl.col("__exercise_id").is_not_null())
        .group_by(["__user_id", "__exercise_id"])
        .agg(pl.len().alias("attempt_count"))
        .filter(
            (~pl.col("__exercise_id").is_in(sorted(exempt_exercise_ids)))
            & (pl.col("attempt_count") > MAX_TOTAL_ATTEMPTS_PER_EXERCISE)
        )
        .select(pl.col("__user_id").alias("user_id"))
        .unique()
    )

    filtered = normalized.join(
        offending_users, left_on="__user_id", right_on="user_id", how="anti"
    ).drop(["__user_id", "__exercise_id"])

    total_students = (
        normalized.select(pl.col("__user_id").drop_nulls().n_unique().alias("value"))
        .collect()
        .item(0, "value")
    )
    total_attempts = normalized.select(pl.len().alias("value")).collect().item(0, "value")
    removed_students = offending_users.select(pl.len().alias("value")).collect().item(0, "value")
    kept_attempts = filtered.select(pl.len().alias("value")).collect().item(0, "value")
    removed_attempts = total_attempts - kept_attempts

    return filtered, {
        "source": source_label,
        "students_removed": removed_students,
        "pct_students_removed": (
            100 * removed_students / total_students if total_students else 0.0
        ),
        "attempts_removed": removed_attempts,
        "pct_attempts_removed": (
            100 * removed_attempts / total_attempts if total_attempts else 0.0
        ),
        "total_students": total_students,
        "total_attempts": total_attempts,
    }


am_exempt_exercise_ids, am_exempt_activity_codes = collect_am_single_exercise_metadata(
    am_learning_catalog,
    AM_SCOPED_MODULE_CODES,
)
adaptiv_world_exempt_exercise_ids, adaptiv_world_exempt_activity_codes = (
    collect_graph_single_exercise_metadata(
        adaptiv_world_graph,
        ADAPTIV_WORLD_SCOPED_MODULE_CODES,
    )
)
adaptiv_college_exempt_exercise_ids, adaptiv_college_exempt_activity_codes = (
    collect_graph_single_exercise_metadata(
        adaptiv_college_graph,
        ADAPTIV_COLLEGE_SCOPED_MODULE_CODES,
    )
)

am_retry_filtered, am_retry_stats = apply_retry_cap(
    am_scoped,
    source_label="am",
    exempt_exercise_ids=am_exempt_exercise_ids,
)
adaptiv_world_retry_filtered, adaptiv_world_retry_stats = apply_retry_cap(
    adaptiv_world_scoped,
    source_label="adaptiv_world",
    exempt_exercise_ids=adaptiv_world_exempt_exercise_ids,
)
adaptiv_college_retry_filtered, adaptiv_college_retry_stats = apply_retry_cap(
    adaptiv_college_scoped,
    source_label="adaptiv_college",
    exempt_exercise_ids=adaptiv_college_exempt_exercise_ids,
)

retry_stats = [
    am_retry_stats,
    adaptiv_world_retry_stats,
    adaptiv_college_retry_stats,
]
retry_filter_summary = pl.DataFrame(retry_stats).with_columns(
    pl.col("pct_students_removed").round(2),
    pl.col("pct_attempts_removed").round(2),
)

total_students = sum(row["total_students"] for row in retry_stats)
total_attempts = sum(row["total_attempts"] for row in retry_stats)
total_removed_students = sum(row["students_removed"] for row in retry_stats)
total_removed_attempts = sum(row["attempts_removed"] for row in retry_stats)
retry_filter_total = pl.DataFrame(
    {
        "source": ["total"],
        "students_removed": [total_removed_students],
        "pct_students_removed": [
            round(100 * total_removed_students / total_students, 2) if total_students else 0.0
        ],
        "attempts_removed": [total_removed_attempts],
        "pct_attempts_removed": [
            round(100 * total_removed_attempts / total_attempts, 2) if total_attempts else 0.0
        ],
        "total_students": [total_students],
        "total_attempts": [total_attempts],
    }
)

print(
    f"Retry cap: more than {MAX_RETRIES_PER_EXERCISE} retries on the same exercise "
    f"(> {MAX_TOTAL_ATTEMPTS_PER_EXERCISE} total attempts) removes the full "
    "source-specific student history."
)
print(
    f"AM exempt single-exercise activities: {len(am_exempt_activity_codes)} "
    f"({len(am_exempt_exercise_ids)} exercise ids)"
)
print(
    "Adaptiv World exempt single-exercise activities: "
    f"{len(adaptiv_world_exempt_activity_codes)} "
    f"({len(adaptiv_world_exempt_exercise_ids)} exercise ids)"
)
print(
    "Adaptiv College exempt single-exercise activities: "
    f"{len(adaptiv_college_exempt_activity_codes)} "
    f"({len(adaptiv_college_exempt_exercise_ids)} exercise ids)"
)
print("Removal summary:")
print(pl.concat([retry_filter_summary, retry_filter_total], how="vertical_relaxed"))

Retry cap: more than 20 retries on the same exercise (> 21 total attempts) removes the full source-specific student history.
AM exempt single-exercise activities: 12 (12 exercise ids)
Adaptiv World exempt single-exercise activities: 0 (0 exercise ids)
Adaptiv College exempt single-exercise activities: 0 (0 exercise ids)
Removal summary:
shape: (4, 7)
┌──────────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┐
│ source       ┆ students_re ┆ pct_student ┆ attempts_re ┆ pct_attempt ┆ total_stude ┆ total_attem │
│ ---          ┆ moved       ┆ s_removed   ┆ moved       ┆ s_removed   ┆ nts         ┆ pts         │
│ str          ┆ ---         ┆ ---         ┆ ---         ┆ ---         ┆ ---         ┆ ---         │
│              ┆ i64         ┆ f64         ┆ i64         ┆ f64         ┆ i64         ┆ i64         │
╞══════════════╪═════════════╪═════════════╪═════════════╪═════════════╪═════════════╪═════════════╡
│ am           ┆ 178         ┆ 0.7       

## Step 6: Normalize columns, derive session and retry fields, and write an intermediate parquet

Cast the retained columns to a shared schema, add the source label (`am`, `adaptiv_world`, or `adaptiv_college`), derive `attempt_index`, and create a pseudonymous `session_id` from each `(source, user_id, login_time)` session. At this stage, `login_time` and the original absolute `created_at` are kept only temporarily so Step 7 can compute the session-relative timestamp.

In [6]:
def prepare_attempts(lf: pl.LazyFrame, *, source_label: str) -> pl.LazyFrame:
    return (
        lf.with_columns(pl.lit(source_label).alias("source"))
        .select(PREP_COLUMNS)
        .with_columns(
            pl.col("user_id").cast(pl.Utf8, strict=False),
            pl.col("classroom_id").cast(pl.Utf8, strict=False),
            pl.col("playlist_or_module_id").cast(pl.Utf8, strict=False),
            pl.col("exercise_id").cast(pl.Utf8, strict=False),
            pl.col("created_at")
            .cast(pl.Utf8, strict=False)
            .str.strptime(pl.Datetime(time_zone="UTC"), format=TIMESTAMP_FORMAT, strict=False),
            pl.col("login_time")
            .cast(pl.Utf8, strict=False)
            .str.strptime(pl.Datetime(time_zone="UTC"), format=TIMESTAMP_FORMAT, strict=False),
            pl.col("data_correct").cast(pl.Boolean, strict=False),
            pl.col("work_mode").cast(pl.Utf8, strict=False),
            pl.col("data_answer").cast(pl.Utf8, strict=False),
            pl.col("data_duration").cast(pl.Float64, strict=False),
            pl.col("source").cast(pl.Utf8, strict=False),
        )
        .sort(["source", "user_id", "exercise_id", "created_at", "login_time"])
        .with_columns(
            pl.col("exercise_id")
            .cum_count()
            .over(["source", "user_id", "exercise_id"])
            .alias("attempt_index"),
            pl.col("login_time")
            .rank("dense")
            .over(["source", "user_id"])
            .cast(pl.UInt32)
            .alias("_session_index"),
        )
        .with_columns(
            pl.when(pl.col("login_time").is_not_null())
            .then(
                pl.concat_str(
                    [
                        pl.col("source"),
                        pl.col("user_id"),
                        pl.concat_str(
                            [pl.lit("session"), pl.col("_session_index").cast(pl.Utf8)],
                            separator="_",
                        ),
                    ],
                    separator="::",
                )
            )
            .otherwise(None)
            .alias("session_id")
        )
        .drop("_session_index")
        .sort(["source", "user_id", "session_id", "created_at", "exercise_id", "attempt_index"])
        .select(INTERMEDIATE_COLUMNS)
    )


maths_data = pl.concat(
    [
        prepare_attempts(am_retry_filtered, source_label="am"),
        prepare_attempts(
            adaptiv_world_retry_filtered,
            source_label="adaptiv_world",
        ),
        prepare_attempts(
            adaptiv_college_retry_filtered,
            source_label="adaptiv_college",
        ),
    ],
    how="vertical_relaxed",
)

MATHS_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
maths_data.sink_parquet(MATHS_DATA_PATH, compression="zstd")

maths_data_preview = pl.scan_parquet(MATHS_DATA_PATH).head(5).collect()
maths_data_rows = (
    pl.scan_parquet(MATHS_DATA_PATH).select(pl.len().alias("rows")).collect().item(0, "rows")
)
source_counts = (
    pl.scan_parquet(MATHS_DATA_PATH)
    .group_by("source")
    .agg(pl.len().alias("rows"))
    .collect()
    .sort("source")
)
max_attempt_index = (
    pl.scan_parquet(MATHS_DATA_PATH)
    .select(pl.col("attempt_index").max().alias("max_attempt_index"))
    .collect()
    .item(0, "max_attempt_index")
)
session_counts = (
    pl.scan_parquet(MATHS_DATA_PATH)
    .group_by("source")
    .agg(pl.col("session_id").drop_nulls().n_unique().alias("sessions"))
    .collect()
    .sort("source")
)

print(f"Saved intermediate merged dataset to: {MATHS_DATA_PATH}")
print(f"Rows written: {maths_data_rows:,}")
print("Rows by source:")
print(source_counts)
print(f"Max attempt_index observed: {max_attempt_index}")
print("Sessions by source:")
print(session_counts)
maths_data_preview

Saved intermediate merged dataset to: C:\Users\ocler\Documents\Académique\Inria\GAIMHE\Code\visu2\data_miaam\maths_data.parquet
Rows written: 5,968,440
Rows by source:
shape: (3, 2)
┌─────────────────┬─────────┐
│ source          ┆ rows    │
│ ---             ┆ ---     │
│ str             ┆ u32     │
╞═════════════════╪═════════╡
│ adaptiv_college ┆ 62650   │
│ adaptiv_world   ┆ 249588  │
│ am              ┆ 5656202 │
└─────────────────┴─────────┘
Max attempt_index observed: 137
Sessions by source:
shape: (3, 2)
┌─────────────────┬──────────┐
│ source          ┆ sessions │
│ ---             ┆ ---      │
│ str             ┆ u32      │
╞═════════════════╪══════════╡
│ adaptiv_college ┆ 2095     │
│ adaptiv_world   ┆ 9380     │
│ am              ┆ 152920   │
└─────────────────┴──────────┘


user_id,classroom_id,playlist_or_module_id,exercise_id,created_at,login_time,data_correct,work_mode,data_answer,data_duration,source,attempt_index,session_id
str,str,str,str,"datetime[μs, UTC]","datetime[μs, UTC]",bool,str,str,f64,str,u32,str
"""00008169-3452-462c-9b68-775c3d…","""db13ffd5-968e-4241-bd14-ce6b2d…","""63e98e5f-94e3-4630-9704-076882…","""77a65f22-ef59-4958-8c0d-a48bf5…",2023-10-16 19:42:44.531 UTC,2023-10-16 18:59:10.256 UTC,true,"""zpdes""","""[1]""",6280.0,"""am""",1,"""am::00008169-3452-462c-9b68-77…"
"""00008169-3452-462c-9b68-775c3d…","""db13ffd5-968e-4241-bd14-ce6b2d…","""63e98e5f-94e3-4630-9704-076882…","""b620b175-7758-48b0-b7da-33ba4a…",2023-10-16 19:42:52.967 UTC,2023-10-16 18:59:10.256 UTC,true,"""zpdes""","""[1]""",3825.0,"""am""",1,"""am::00008169-3452-462c-9b68-77…"
"""00008169-3452-462c-9b68-775c3d…","""db13ffd5-968e-4241-bd14-ce6b2d…","""63e98e5f-94e3-4630-9704-076882…","""f1319fb1-492d-4f70-932a-b3816d…",2023-10-16 19:42:57.118 UTC,2023-10-16 18:59:10.256 UTC,true,"""zpdes""","""[0]""",2472.0,"""am""",1,"""am::00008169-3452-462c-9b68-77…"
"""00008169-3452-462c-9b68-775c3d…","""db13ffd5-968e-4241-bd14-ce6b2d…","""63e98e5f-94e3-4630-9704-076882…","""6e0207af-55b5-498d-990b-e1365b…",2023-10-16 19:43:01.872 UTC,2023-10-16 18:59:10.256 UTC,true,"""zpdes""","""[1]""",3278.0,"""am""",1,"""am::00008169-3452-462c-9b68-77…"
"""00008169-3452-462c-9b68-775c3d…","""db13ffd5-968e-4241-bd14-ce6b2d…","""63e98e5f-94e3-4630-9704-076882…","""91864db5-1430-4a98-8dee-22cb7a…",2023-10-16 19:43:06.607 UTC,2023-10-16 18:59:10.256 UTC,true,"""zpdes""","""[1]""",3333.0,"""am""",1,"""am::00008169-3452-462c-9b68-77…"


## Step 7: Replace absolute timestamps with a session-relative clock

For rows with `login_time`, convert `created_at` to the elapsed time since the original `login_time` within the same session. The login date is kept, but the login clock time becomes `00:00:00`: for example, if `login_time` was 08:00 and the original attempt time was 08:01, the released `created_at` becomes 00:01 on that login date.

For rows where `login_time` is missing, keep `session_id` missing and use a conservative fallback: within each `(source, user_id, calendar day)`, set the first observed attempt to `00:00:00` and express later attempts on that day relative to that first observed attempt. After this conversion, `login_time` and the original absolute `created_at` are not kept in the parquet.


In [7]:
raw_day = pl.col("created_at").dt.replace_time_zone(None).dt.truncate("1d")
first_observed_created_at_for_fallback_group = (
    pl.col("created_at").min().over(["source", "user_id", "_raw_day", "_missing_login_time"])
)

login_relative_created_at = pl.col("login_time").dt.replace_time_zone(None).dt.truncate(
    "1d"
) + pl.duration(
    microseconds=(pl.col("created_at").dt.epoch("us") - pl.col("login_time").dt.epoch("us"))
)
fallback_day_relative_created_at = pl.col("_raw_day") + pl.duration(
    microseconds=(
        pl.col("created_at").dt.epoch("us")
        - first_observed_created_at_for_fallback_group.dt.epoch("us")
    )
)

maths_data_with_session_clock = (
    pl.scan_parquet(MATHS_DATA_PATH)
    .with_columns(raw_day.alias("_raw_day"))
    .with_columns(pl.col("login_time").is_null().alias("_missing_login_time"))
    .with_columns(
        pl.when(pl.col("login_time").is_not_null())
        .then(login_relative_created_at)
        .otherwise(fallback_day_relative_created_at)
        .alias("created_at")
    )
    .select(FINAL_COLUMNS)
)

sink_parquet_replace(maths_data_with_session_clock, MATHS_DATA_PATH, compression="zstd")

session_clock_preview = (
    pl.scan_parquet(MATHS_DATA_PATH)
    .select(
        [
            "source",
            "user_id",
            "classroom_id",
            "session_id",
            "created_at",
            "attempt_index",
        ]
    )
    .head(5)
    .collect()
)
final_timestamp_columns = pl.scan_parquet(MATHS_DATA_PATH).collect_schema().names()
null_session_created_at = (
    pl.scan_parquet(MATHS_DATA_PATH)
    .filter(pl.col("session_id").is_null())
    .select(pl.col("created_at").is_null().sum().alias("null_created_at_for_null_session_rows"))
    .collect()
    .item(0, "null_created_at_for_null_session_rows")
)
removed_absolute_time_columns = ("login_time" not in final_timestamp_columns) and (
    "created_at_session_time" not in final_timestamp_columns
)

print(f"Rewrote {MATHS_DATA_PATH} with anonymized created_at values.")
print(
    f"login_time and created_at_session_time absent from final schema: {removed_absolute_time_columns}"
)
print(f"Rows with missing session_id and missing created_at: {null_session_created_at}")
session_clock_preview

Rewrote C:\Users\ocler\Documents\Académique\Inria\GAIMHE\Code\visu2\data_miaam\maths_data.parquet with anonymized created_at values.
login_time and created_at_session_time absent from final schema: True
Rows with missing session_id and missing created_at: 0


source,user_id,classroom_id,session_id,created_at,attempt_index
str,str,str,str,datetime[μs],u32
"""am""","""00008169-3452-462c-9b68-775c3d…","""db13ffd5-968e-4241-bd14-ce6b2d…","""am::00008169-3452-462c-9b68-77…",2023-10-16 00:43:34.275,1
"""am""","""00008169-3452-462c-9b68-775c3d…","""db13ffd5-968e-4241-bd14-ce6b2d…","""am::00008169-3452-462c-9b68-77…",2023-10-16 00:43:42.711,1
"""am""","""00008169-3452-462c-9b68-775c3d…","""db13ffd5-968e-4241-bd14-ce6b2d…","""am::00008169-3452-462c-9b68-77…",2023-10-16 00:43:46.862,1
"""am""","""00008169-3452-462c-9b68-775c3d…","""db13ffd5-968e-4241-bd14-ce6b2d…","""am::00008169-3452-462c-9b68-77…",2023-10-16 00:43:51.616,1
"""am""","""00008169-3452-462c-9b68-775c3d…","""db13ffd5-968e-4241-bd14-ce6b2d…","""am::00008169-3452-462c-9b68-77…",2023-10-16 00:43:56.351,1


## Step 8: Report borderline adaptive-test transition cases without filtering

Compute the adaptive-test edge cases discussed for schema review on the current merged dataset after the earlier scope and retry preprocessing. Sessions are ordered by the numeric suffix of the pseudonymous `session_N` identifier so that `session_10` cannot sort before `session_2`. The cell only prints diagnostics; it does not remove anything. Student counts are based on `(source, user_id)` pairs.


In [8]:
from collections import defaultdict

try:
    import pandas as pd
except Exception:
    pd = None

try:
    from IPython.display import HTML, Markdown, display
except Exception:
    HTML = None
    Markdown = None

    def display(value):
        print(value)


def display_note(text: str) -> None:
    if Markdown is not None:
        display(Markdown(text))
    else:
        print(text)


def display_table(title: str, explanation: str, frame: pl.DataFrame) -> None:
    display_note(f"### {title}\n\n{explanation}")
    if HTML is not None and pd is not None:
        html = frame.to_pandas().to_html(index=False)
        wrapped_html = (
            "<div style='max-width: 1600px; overflow-x: auto;'>"
            "<style>"
            "table.dataframe {border-collapse: collapse; width: 100%;}"
            "table.dataframe th, table.dataframe td {white-space: normal; word-break: break-word; vertical-align: top; padding: 6px 10px; text-align: left;}"
            "table.dataframe th {font-weight: 600;}"
            "</style>"
            f"{html}</div>"
        )
        display(HTML(wrapped_html))
    else:
        with pl.Config(tbl_rows=-1, tbl_cols=-1, fmt_str_lengths=200):
            print(frame)


BORDERLINE_MIN_ADAPTIVE_ATTEMPTS = 5

module_lookup_rows: list[dict[str, str]] = []
for module in am_learning_catalog["modules"]:
    module_code = str(module.get("code") or "")
    if module_code in set(AM_SCOPED_MODULE_CODES):
        module_lookup_rows.append(
            {
                "source": "am",
                "playlist_or_module_id": str(module["id"]),
                "module_code": module_code,
            }
        )
for source_label, graph, scoped_module_codes in (
    (
        "adaptiv_world",
        adaptiv_world_graph,
        ADAPTIV_WORLD_SCOPED_MODULE_CODES,
    ),
    (
        "adaptiv_college",
        adaptiv_college_graph,
        ADAPTIV_COLLEGE_SCOPED_MODULE_CODES,
    ),
):
    for module in graph["modules"]:
        module_code = str(module.get("code") or "")
        if module_code in set(scoped_module_codes):
            module_lookup_rows.append(
                {
                    "source": source_label,
                    "playlist_or_module_id": str(module["id"]),
                    "module_code": module_code,
                }
            )
module_lookup = pl.DataFrame(module_lookup_rows).unique()

borderline_base = (
    pl.scan_parquet(MATHS_DATA_PATH)
    .select(
        [
            "source",
            "user_id",
            "playlist_or_module_id",
            "created_at",
            "session_id",
            "work_mode",
            "exercise_id",
            "attempt_index",
        ]
    )
    .with_columns(
        pl.col("session_id")
        .str.extract(r"::session_(\d+)$", 1)
        .cast(pl.UInt32, strict=False)
        .alias("_session_order")
    )
    .join(module_lookup.lazy(), on=["source", "playlist_or_module_id"], how="left")
    .with_columns(
        pl.concat_str(["source", "user_id"], separator="::").alias("student_key"),
        pl.coalesce([pl.col("module_code"), pl.col("playlist_or_module_id")]).alias("module_code"),
    )
    .select(
        [
            "student_key",
            "source",
            "user_id",
            "created_at",
            "session_id",
            "_session_order",
            "work_mode",
            "module_code",
            "exercise_id",
            "attempt_index",
        ]
    )
)

invalid_session_order_rows = (
    borderline_base.filter(
        pl.col("session_id").is_not_null() & pl.col("_session_order").is_null()
    )
    .select(pl.len().alias("rows"))
    .collect()
    .item(0, "rows")
)
assert invalid_session_order_rows == 0, (
    f"Could not recover numeric order from {invalid_session_order_rows} session ids"
)

borderline_student_totals = (
    borderline_base.group_by(["student_key", "source", "user_id"])
    .agg(pl.len().alias("attempts"))
    .collect()
)
borderline_segment_summary = (
    borderline_base.sort(
        ["student_key", "_session_order", "created_at", "module_code", "exercise_id", "attempt_index"]
    )
    .with_columns(
        (
            pl.col("student_key").ne(pl.col("student_key").shift(1)).fill_null(True)
            | pl.col("work_mode").ne(pl.col("work_mode").shift(1)).fill_null(True)
        ).alias("_segment_start")
    )
    .with_columns(
        pl.col("_segment_start").cast(pl.Int64).cum_sum().over("student_key").alias("segment_index")
    )
    .group_by(["student_key", "source", "user_id", "segment_index"], maintain_order=True)
    .agg(
        pl.first("work_mode").alias("segment_work_mode"),
        pl.len().alias("segment_attempts"),
        pl.col("module_code").drop_nulls().unique().sort().alias("segment_module_codes"),
    )
    .sort(["student_key", "segment_index"])
    .collect()
)

student_attempt_map = {
    row["student_key"]: int(row["attempts"])
    for row in borderline_student_totals.iter_rows(named=True)
}
student_source_map = {
    row["student_key"]: row["source"] for row in borderline_student_totals.iter_rows(named=True)
}
totals_by_source = {
    row["source"]: {"students": int(row["students"]), "attempts": int(row["attempts"])}
    for row in borderline_student_totals.group_by("source")
    .agg(pl.len().alias("students"), pl.sum("attempts").alias("attempts"))
    .iter_rows(named=True)
}
total_students = borderline_student_totals.height
total_attempts = int(borderline_student_totals.get_column("attempts").sum())

segments_by_student: dict[str, list[dict[str, object]]] = defaultdict(list)
segment_attempts_map: dict[tuple[str, int], int] = {}
segment_source_map: dict[tuple[str, int], str] = {}
for row in borderline_segment_summary.iter_rows(named=True):
    segments_by_student[row["student_key"]].append(row)
    key = (row["student_key"], int(row["segment_index"]))
    segment_attempts_map[key] = int(row["segment_attempts"])
    segment_source_map[key] = str(row["source"])

adaptive_multimodule_all_new_students: set[str] = set()
adaptive_multimodule_all_new_segments: set[tuple[str, int]] = set()
adaptive_multimodule_with_prior_students: set[str] = set()
adaptive_same_module_repeat_students: set[str] = set()
adaptive_short_block_students: set[str] = set()
adaptive_short_block_segments: set[tuple[str, int]] = set()
adaptive_short_plus_following_segments: set[tuple[str, int]] = set()

for student_key, segments in segments_by_student.items():
    previous_zpdes_modules: set[str] = set()
    repeated_single_module_counts: dict[str, int] = defaultdict(int)
    for idx, segment in enumerate(segments):
        next_segment = segments[idx + 1] if idx + 1 < len(segments) else None
        if segment["segment_work_mode"] == "adaptive-test":
            if int(segment["segment_attempts"]) < BORDERLINE_MIN_ADAPTIVE_ATTEMPTS:
                adaptive_short_block_students.add(student_key)
                adaptive_short_block_segments.add((student_key, int(segment["segment_index"])))
                if next_segment and next_segment["segment_work_mode"] == "zpdes":
                    adaptive_short_plus_following_segments.add(
                        (student_key, int(segment["segment_index"]))
                    )
                    adaptive_short_plus_following_segments.add(
                        (student_key, int(next_segment["segment_index"]))
                    )
            if next_segment and next_segment["segment_work_mode"] == "zpdes":
                next_modules = set(next_segment["segment_module_codes"] or [])
                if len(next_modules) > 1:
                    if next_modules.isdisjoint(previous_zpdes_modules):
                        adaptive_multimodule_all_new_students.add(student_key)
                        adaptive_multimodule_all_new_segments.add(
                            (student_key, int(segment["segment_index"]))
                        )
                        adaptive_multimodule_all_new_segments.add(
                            (student_key, int(next_segment["segment_index"]))
                        )
                    else:
                        adaptive_multimodule_with_prior_students.add(student_key)
                if len(next_modules) == 1:
                    repeated_single_module_counts[next(iter(next_modules))] += 1
        if segment["segment_work_mode"] == "zpdes":
            previous_zpdes_modules.update(segment["segment_module_codes"] or [])
    if any(count > 1 for count in repeated_single_module_counts.values()):
        adaptive_same_module_repeat_students.add(student_key)


def summarize_student_case(case_label: str, students: set[str]) -> list[dict[str, object]]:
    rows: list[dict[str, object]] = []
    for source in SOURCE_LABELS:
        source_students = [student for student in students if student_source_map[student] == source]
        student_count = len(source_students)
        attempt_count = sum(student_attempt_map[student] for student in source_students)
        source_totals = totals_by_source[source]
        rows.append(
            {
                "case": case_label,
                "source": source,
                "students": student_count,
                "pct_students": round(100 * student_count / source_totals["students"], 2)
                if source_totals["students"]
                else 0.0,
                "attempts": attempt_count,
                "pct_attempts": round(100 * attempt_count / source_totals["attempts"], 2)
                if source_totals["attempts"]
                else 0.0,
            }
        )
    rows.append(
        {
            "case": case_label,
            "source": "total",
            "students": len(students),
            "pct_students": round(100 * len(students) / total_students, 2)
            if total_students
            else 0.0,
            "attempts": sum(student_attempt_map[student] for student in students),
            "pct_attempts": round(
                100 * sum(student_attempt_map[student] for student in students) / total_attempts, 2
            )
            if total_attempts
            else 0.0,
        }
    )
    return rows


def summarize_segment_case(
    case_label: str, segment_keys: set[tuple[str, int]]
) -> list[dict[str, object]]:
    rows: list[dict[str, object]] = []
    affected_students = {student_key for student_key, _segment_index in segment_keys}
    for source in SOURCE_LABELS:
        source_keys = [key for key in segment_keys if segment_source_map[key] == source]
        source_students = {student_key for student_key, _segment_index in source_keys}
        attempt_count = sum(segment_attempts_map[key] for key in source_keys)
        source_totals = totals_by_source[source]
        rows.append(
            {
                "case": case_label,
                "source": source,
                "students": len(source_students),
                "pct_students": round(100 * len(source_students) / source_totals["students"], 2)
                if source_totals["students"]
                else 0.0,
                "attempts": attempt_count,
                "pct_attempts": round(100 * attempt_count / source_totals["attempts"], 2)
                if source_totals["attempts"]
                else 0.0,
            }
        )
    rows.append(
        {
            "case": case_label,
            "source": "total",
            "students": len(affected_students),
            "pct_students": round(100 * len(affected_students) / total_students, 2)
            if total_students
            else 0.0,
            "attempts": sum(segment_attempts_map[key] for key in segment_keys),
            "pct_attempts": round(
                100 * sum(segment_attempts_map[key] for key in segment_keys) / total_attempts, 2
            )
            if total_attempts
            else 0.0,
        }
    )
    return rows


borderline_whole_student_summary = pl.DataFrame(
    summarize_student_case(
        "Adaptive -> multi-module ZPDES, all modules unseen before (remove whole student)",
        adaptive_multimodule_all_new_students,
    )
    + summarize_student_case(
        "Adaptive -> multi-module ZPDES, at least one module seen before (remove whole student)",
        adaptive_multimodule_with_prior_students,
    )
    + summarize_student_case(
        "Adaptive -> ZPDES(single module M) repeated for the same M (remove whole student)",
        adaptive_same_module_repeat_students,
    )
    + summarize_student_case(
        "Adaptive block with <5 attempts (remove whole student)",
        adaptive_short_block_students,
    )
).sort(["case", "source"])

borderline_segment_summary_output = pl.DataFrame(
    summarize_segment_case(
        "Adaptive block with <5 attempts (drop short adaptive blocks only)",
        adaptive_short_block_segments,
    )
    + summarize_segment_case(
        "Adaptive block with <5 attempts (drop short adaptive + following ZPDES)",
        adaptive_short_plus_following_segments,
    )
).sort(["case", "source"])

baseline_summary = pl.DataFrame(
    [
        {"source": source, "students": values["students"], "attempts": values["attempts"]}
        for source, values in sorted(totals_by_source.items())
    ]
    + [{"source": "total", "students": total_students, "attempts": total_attempts}]
)

display_note(
    "## Borderline case diagnostics\n\nThese tables describe the current merged maths dataset after the earlier scope and retry preprocessing. Student counts are based on `(source, user_id)` pairs."
)

display_table(
    "Baseline used for the diagnostics",
    "This table gives the denominator used for the percentages below: the number of retained students and attempts in each source before applying any new borderline-case decision rule.",
    baseline_summary,
)

display_table(
    "Whole-student borderline cases",
    "Each row shows what would be removed if we decided to exclude the full history of students matching that adaptive-test pattern.",
    borderline_whole_student_summary,
)

display_table(
    "Segment-level variants for short adaptive blocks",
    "These rows quantify two lighter alternatives for short adaptive blocks: dropping only the short adaptive block itself, or dropping that block plus the ZPDES block that comes immediately after it.",
    borderline_segment_summary_output,
)

## Borderline case diagnostics

These tables describe the current merged maths dataset after the earlier scope and retry preprocessing. Student counts are based on `(source, user_id)` pairs.

### Baseline used for the diagnostics

This table gives the denominator used for the percentages below: the number of retained students and attempts in each source before applying any new borderline-case decision rule.

source,students,attempts
adaptiv_college,911,62650
adaptiv_world,1410,249588
am,25309,5656202
total,27630,5968440


### Whole-student borderline cases

Each row shows what would be removed if we decided to exclude the full history of students matching that adaptive-test pattern.

case,source,students,pct_students,attempts,pct_attempts
Adaptive -> ZPDES(single module M) repeated for the same M (remove whole student),adaptiv_college,32,3.51,4697,7.50
Adaptive -> ZPDES(single module M) repeated for the same M (remove whole student),adaptiv_world,40,2.84,7925,3.18
Adaptive -> ZPDES(single module M) repeated for the same M (remove whole student),am,935,3.69,458764,8.11
Adaptive -> ZPDES(single module M) repeated for the same M (remove whole student),total,1007,3.64,471386,7.90
"Adaptive -> multi-module ZPDES, all modules unseen before (remove whole student)",adaptiv_college,13,1.43,2348,3.75
"Adaptive -> multi-module ZPDES, all modules unseen before (remove whole student)",adaptiv_world,265,18.79,87414,35.02
"Adaptive -> multi-module ZPDES, all modules unseen before (remove whole student)",am,16,0.06,4629,0.08
"Adaptive -> multi-module ZPDES, all modules unseen before (remove whole student)",total,294,1.06,94391,1.58
"Adaptive -> multi-module ZPDES, at least one module seen before (remove whole student)",adaptiv_college,34,3.73,8015,12.79
"Adaptive -> multi-module ZPDES, at least one module seen before (remove whole student)",adaptiv_world,48,3.40,15673,6.28


### Segment-level variants for short adaptive blocks

These rows quantify two lighter alternatives for short adaptive blocks: dropping only the short adaptive block itself, or dropping that block plus the ZPDES block that comes immediately after it.

case,source,students,pct_students,attempts,pct_attempts
Adaptive block with <5 attempts (drop short adaptive + following ZPDES),adaptiv_college,13,1.43,327,0.52
Adaptive block with <5 attempts (drop short adaptive + following ZPDES),adaptiv_world,96,6.81,9986,4.00
Adaptive block with <5 attempts (drop short adaptive + following ZPDES),am,672,2.66,116040,2.05
Adaptive block with <5 attempts (drop short adaptive + following ZPDES),total,781,2.83,126353,2.12
Adaptive block with <5 attempts (drop short adaptive blocks only),adaptiv_college,126,13.83,295,0.47
Adaptive block with <5 attempts (drop short adaptive blocks only),adaptiv_world,189,13.40,585,0.23
Adaptive block with <5 attempts (drop short adaptive blocks only),am,1568,6.20,4627,0.08
Adaptive block with <5 attempts (drop short adaptive blocks only),total,1883,6.82,5507,0.09


## Step 9: Apply the selected adaptive borderline filters

Remove the same three patterns as the original notebook: whole students with repeated `adaptive-test -> zpdes(M)` for the same single module, short adaptive blocks plus their following ZPDES block when the adaptive block has fewer than five attempts, and adaptive blocks that lead to a multi-module ZPDES block where all modules are unseen before, together with that following ZPDES block. Detection, filtering, and `attempt_index` recomputation all use the numeric session order recovered from `session_N`; the public `session_id` format remains unchanged. The filtered dataset is written back to `data_miaam/maths_data.parquet`.

In [9]:
required_step8_names = [
    "module_lookup",
    "adaptive_same_module_repeat_students",
    "adaptive_short_plus_following_segments",
    "adaptive_multimodule_all_new_segments",
]
missing_step8_names = [name for name in required_step8_names if name not in globals()]
if missing_step8_names:
    missing_text = ", ".join(sorted(missing_step8_names))
    raise RuntimeError(f"Run Step 8 before Step 9. Missing variables: {missing_text}")

FILTERED_WHOLE_STUDENTS = sorted(adaptive_same_module_repeat_students)
FILTERED_SEGMENT_KEYS = sorted(
    set(adaptive_short_plus_following_segments) | set(adaptive_multimodule_all_new_segments)
)

whole_student_keys_df = pl.DataFrame(
    {"student_key": FILTERED_WHOLE_STUDENTS},
    schema={"student_key": pl.Utf8},
)
segment_keys_df = pl.DataFrame(
    {
        "student_key": [student_key for student_key, _segment_index in FILTERED_SEGMENT_KEYS],
        "segment_index": [
            int(segment_index) for _student_key, segment_index in FILTERED_SEGMENT_KEYS
        ],
    },
    schema={"student_key": pl.Utf8, "segment_index": pl.Int64},
)

pre_filter_schema = pl.scan_parquet(MATHS_DATA_PATH).collect_schema()
pre_filter_columns = pre_filter_schema.names()

rows_with_segments = (
    pl.scan_parquet(MATHS_DATA_PATH)
    .join(module_lookup.lazy(), on=["source", "playlist_or_module_id"], how="left")
    .with_columns(
        pl.concat_str(["source", "user_id"], separator="::").alias("student_key"),
        pl.coalesce([pl.col("module_code"), pl.col("playlist_or_module_id")]).alias("module_code"),
        pl.col("session_id")
        .str.extract(r"::session_(\d+)$", 1)
        .cast(pl.UInt32, strict=False)
        .alias("_session_order"),
    )
    .sort(
        ["student_key", "_session_order", "created_at", "module_code", "exercise_id", "attempt_index"]
    )
    .with_columns(
        (
            pl.col("student_key").ne(pl.col("student_key").shift(1)).fill_null(True)
            | pl.col("work_mode").ne(pl.col("work_mode").shift(1)).fill_null(True)
        ).alias("_segment_start")
    )
    .with_columns(
        pl.col("_segment_start").cast(pl.Int64).cum_sum().over("student_key").alias("segment_index")
    )
)

after_whole_student_filter = rows_with_segments.join(
    whole_student_keys_df.lazy(), on="student_key", how="anti"
)
after_segment_filter = after_whole_student_filter.join(
    segment_keys_df.lazy(), on=["student_key", "segment_index"], how="anti"
)

recomputed_attempts = (
    after_segment_filter.sort(
        [
            "source",
            "user_id",
            "exercise_id",
            "_session_order",
            "created_at",
            "attempt_index",
        ]
    )
    .with_columns(
        pl.col("exercise_id")
        .cum_count()
        .over(["source", "user_id", "exercise_id"])
        .alias("attempt_index")
    )
    .sort(
        ["source", "user_id", "_session_order", "created_at", "exercise_id", "attempt_index"]
    )
    .drop(["student_key", "module_code", "_session_order", "_segment_start", "segment_index"])
)

final_filtered = recomputed_attempts.select(pre_filter_columns)


def stage_counts(lf: pl.LazyFrame, label: str) -> pl.DataFrame:
    per_source = (
        lf.group_by("source")
        .agg(
            pl.col("student_key").n_unique().alias("students"),
            pl.len().alias("attempts"),
        )
        .sort("source")
        .collect()
    )
    total_row = pl.DataFrame(
        {
            "source": ["total"],
            "students": [
                lf.select(pl.col("student_key").n_unique().alias("value"))
                .collect()
                .item(0, "value")
            ],
            "attempts": [lf.select(pl.len().alias("value")).collect().item(0, "value")],
        }
    )
    return (
        pl.concat([per_source, total_row], how="vertical_relaxed")
        .with_columns(pl.lit(label).alias("stage"))
        .select(["stage", "source", "students", "attempts"])
    )


stage_summary_table = pl.concat(
    [
        stage_counts(rows_with_segments.select(["student_key", "source"]), "Before Step 9 filters"),
        stage_counts(
            after_whole_student_filter.select(["student_key", "source"]),
            "After repeated adaptive -> ZPDES(M) whole-student removal",
        ),
        stage_counts(
            after_segment_filter.select(["student_key", "source"]),
            "After segment removals (short adaptive + case A)",
        ),
    ],
    how="vertical_relaxed",
)

applied_rules_table = pl.DataFrame(
    [
        {
            "rule": "Remove whole student for repeated adaptive -> ZPDES(single module M) for the same M",
            "items_flagged": len(FILTERED_WHOLE_STUDENTS),
            "unit": "students",
        },
        {
            "rule": "Remove short adaptive blocks (<5 attempts) plus the following ZPDES block",
            "items_flagged": len(adaptive_short_plus_following_segments) // 2,
            "unit": "adaptive->zpdes block pairs",
        },
        {
            "rule": "Remove adaptive -> multi-module ZPDES(all unseen before) plus the following ZPDES block",
            "items_flagged": len(adaptive_multimodule_all_new_segments) // 2,
            "unit": "adaptive->zpdes block pairs",
        },
    ]
)


def format_int(value: int) -> str:
    return f"{int(value):,}"


baseline_total_students = int(
    rows_with_segments.select(pl.col("student_key").n_unique().alias("value"))
    .collect()
    .item(0, "value")
)
baseline_total_attempts = int(
    rows_with_segments.select(pl.len().alias("value")).collect().item(0, "value")
)
final_total_students = int(
    after_segment_filter.select(pl.col("student_key").n_unique().alias("value"))
    .collect()
    .item(0, "value")
)
final_total_attempts = int(
    after_segment_filter.select(pl.len().alias("value")).collect().item(0, "value")
)
final_distinct_exercises = int(
    after_segment_filter.select(pl.col("exercise_id").drop_nulls().n_unique().alias("value"))
    .collect()
    .item(0, "value")
)
filtered_student_pct = (
    100 * (baseline_total_students - final_total_students) / baseline_total_students
    if baseline_total_students
    else 0.0
)
filtered_attempt_pct = (
    100 * (baseline_total_attempts - final_total_attempts) / baseline_total_attempts
    if baseline_total_attempts
    else 0.0
)

step9_summary_lines = (
    f"- {filtered_student_pct:.2f}% of students were filtered out, and {filtered_attempt_pct:.2f}% of attempts were filtered out.\n"
    f"- Final dataset: {format_int(final_total_students)} students, {format_int(final_total_attempts)} attempts, across {format_int(final_distinct_exercises)} distinct exercises."
)

sink_parquet_replace(final_filtered, MATHS_DATA_PATH, compression="zstd")

display_note(
    "## Step 9 results\n\nThis step applies the three selected adaptive borderline filters to the current merged dataset, rewrites `maths_data.parquet`, and recomputes `attempt_index` on the retained rows while preserving the login-time-relative `created_at` values.\n\n"
    + step9_summary_lines
)
display_table(
    "Rules applied in Step 9",
    "This table lists the three filter rules that were applied and how many students or adaptive-to-ZPDES block pairs were flagged before the actual rewrite.",
    applied_rules_table,
)
display_table(
    "Stage-by-stage retained counts",
    "This table shows how many students and attempts remain after the whole-student removal stage and after the two segment-removal stages combined.",
    stage_summary_table,
)
print(f"Rewrote filtered dataset to: {MATHS_DATA_PATH}")

## Step 9 results

This step applies the three selected adaptive borderline filters to the current merged dataset, rewrites `maths_data.parquet`, and recomputes `attempt_index` on the retained rows while preserving the login-time-relative `created_at` values.

- 4.81% of students were filtered out, and 9.65% of attempts were filtered out.
- Final dataset: 26,302 students, 5,392,610 attempts, across 9,427 distinct exercises.

### Rules applied in Step 9

This table lists the three filter rules that were applied and how many students or adaptive-to-ZPDES block pairs were flagged before the actual rewrite.

rule,items_flagged,unit
Remove whole student for repeated adaptive -> ZPDES(single module M) for the same M,1007,students
Remove short adaptive blocks (<5 attempts) plus the following ZPDES block,1156,adaptive->zpdes block pairs
Remove adaptive -> multi-module ZPDES(all unseen before) plus the following ZPDES block,294,adaptive->zpdes block pairs


### Stage-by-stage retained counts

This table shows how many students and attempts remain after the whole-student removal stage and after the two segment-removal stages combined.

stage,source,students,attempts
Before Step 9 filters,adaptiv_college,911,62650
Before Step 9 filters,adaptiv_world,1410,249588
Before Step 9 filters,am,25309,5656202
Before Step 9 filters,total,27630,5968440
After repeated adaptive -> ZPDES(M) whole-student removal,adaptiv_college,879,57953
After repeated adaptive -> ZPDES(M) whole-student removal,adaptiv_world,1370,241663
After repeated adaptive -> ZPDES(M) whole-student removal,am,24374,5197438
After repeated adaptive -> ZPDES(M) whole-student removal,total,26623,5497054
After segment removals (short adaptive + case A),adaptiv_college,867,55680
After segment removals (short adaptive + case A),adaptiv_world,1075,148846


Rewrote filtered dataset to: C:\Users\ocler\Documents\Académique\Inria\GAIMHE\Code\visu2\data_miaam\maths_data.parquet
